In [5]:
import numpy as np
import pandas as pd
import tensorflow as tf
import random
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import glob

# ✅ 시드 고정
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# 시퀀스 길이 (예: 과거 30일 데이터를 보고 다음 날 예측)
sequence_length = 60

# 기술적 지표 목록
technical_indicators = [
    "sma_50", "ema_20", "wma_20", "macd", "macd_signal", "macd_hist", "tema_20",
    "rsi_14", "roc", "cci_14", "willr_14", "atr_14", "upper_bb", "middle_bb", "lower_bb",
    "obv", "ad", "chaikin_ad"
]

# 데이터 불러오기
csv_files = glob.glob("*_Train.csv")
df = pd.read_csv(csv_files[0])  # BTC 데이터 사용

# 독립 변수(X)와 종속 변수(y) 분리
X = df[technical_indicators].values
y = df["label"].values

# 🔹 라벨 값 변경 (-1 → 2, 0 → 0, 1 → 1)
y = y + 1

# 데이터 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 시계열 데이터로 변환
def create_sequences(X, y, seq_length):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i : i + seq_length])
        ys.append(y[i + seq_length])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y, sequence_length)

# 훈련/테스트 데이터 분할 (80:20 비율, ✅ 시드 고정)
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=SEED, stratify=y_seq
)

# ✅ 시드 고정 (TensorFlow 실행)
tf.random.set_seed(SEED)

# GRU 모델 생성
gru_model = Sequential([
    GRU(64, return_sequences=True, input_shape=(sequence_length, len(technical_indicators))),
    Dropout(0.2),
    GRU(32),
    Dropout(0.2),
    Dense(3, activation="softmax")  # 3 클래스 분류
])

# 모델 컴파일 및 학습
gru_model.compile(optimizer=Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_gru = gru_model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_test, y_test))

# 🔹 학습 데이터 저장
train_loss_gru = history_gru.history['loss'][-1]
train_accuracy_gru = history_gru.history['accuracy'][-1]

# 모델 저장
gru_model.save("GRU_model.h5")
print("GRU 모델 저장 완료!")


Epoch 1/50


C:\Users\marigold\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.3563 - loss: 1.1047 - val_accuracy: 0.3262 - val_loss: 1.1090
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.3765 - loss: 1.0961 - val_accuracy: 0.3262 - val_loss: 1.1062
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3857 - loss: 1.0872 - val_accuracy: 0.3262 - val_loss: 1.1078
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.3817 - loss: 1.0836 - val_accuracy: 0.3298 - val_loss: 1.1117
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.3749 - loss: 1.0897 - val_accuracy: 0.3511 - val_loss: 1.1115
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.3988 - loss: 1.0725 - val_accuracy: 0.3475 - val_loss: 1.1135
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4160 - loss: 1.0715 - val_accuracy: 0.3369 - val_loss: 1.1147
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4017 - loss: 1.0799 - val_accuracy: 0.3121 - val_loss: 1.

GRU 모델 저장 완료!
